In [ ]:
!pip install geopandas shapely geohash2
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box, Point
from shapely import wkt
import geohash2
from shapely.wkt import loads

In [ ]:
df = pd.read_csv("problems.csv")
df

,Boro,Vol,geometry,congestion,congestion_level,timestamp,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Street Name
0,Queens,80,POLYGON ((-73.81541389765073 40.74737074495193...,0.035500,0,2024-06-08 22:30:00,New York City Police Department,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,OAK AVENUE
1,Bronx,69,POLYGON ((-73.84214963396792 40.85815387505401...,0.122636,0,2025-04-21 11:15:00,New York City Police Department,Blocked Driveway,Partial Access,Street/Sidewalk,EASTCHESTER ROAD
2,Manhattan,223,"POLYGON ((-73.9743296272132 40.78160532554132,...",0.294327,0,2025-06-21 12:30:00,New York City Police Department,Illegal Parking,Blocked Bike Lane,Street/Sidewalk,COLUMBUS AVENUE
3,Manhattan,72,"POLYGON ((-73.9743296272132 40.78160532554132,...",-1.557482,0,2025-06-22 06:30:00,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,COLUMBUS AVENUE
4,Manhattan,115,POLYGON ((-73.98885407294881 40.77188224822505...,0.523082,0,2024-10-16 15:30:00,Department of Transportation,Street Condition,Pothole,NaN,WEST END AVENUE
...,...,...,...,...,...,...,...,...,...,...,...
258,Brooklyn,234,POLYGON ((-73.93601997464899 40.72387248144743...,0.721558,0,2024-03-16 13:00:00,Department of Transportation,Traffic Signal Condition,Enclosure Cap,NaN,NaN
259,Queens,57,POLYGON ((-73.80240146876282 40.68797242598935...,3.000000,2,2024-01-23 15:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET
260,Queens,39,POLYGON ((-73.80240146876282 40.68797242598935...,1.939188,1,2024-01-23 17:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET
261,Queens,39,POLYGON ((-73.80240146876282 40.68797242598935...,1.939188,1,2024-01-23 17:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET


In [ ]:
!pip install transformers accelerate torch
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
import random
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
from openai import OpenAI
from huggingface_hub import login

os.environ["HF_TOKEN"] = "add your hugging face token"
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

def generate_tweets_api(row):
    n = random.randint(1, 10)

    prompt = f"""
You are a regular person posting on Twitter.

Context:
Street: {row['Street Name']}
Issue: {row['Problem (formerly Complaint Type)']}
Detail: {row['Problem Detail (formerly Descriptor)']}
Congestion Level: {row['congestion_level']} (0 = low, 1 = medium, 2 = high; value is always between 0 and 2)

Write {n} different short tweets under 200 characters.
Casual tone. First person. Do NOT mention all details every time.
No hashtags. No emojis.
Separate each tweet with ||.
Only output the tweets.
"""

    completion = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct:novita",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9,
        max_tokens=200,
    )

    text = completion.choices[0].message.content
    tweets = [t.strip() for t in text.split("||") if t.strip()]
    return tweets


tweets_list = []

for i, row in df.iterrows():
    tweets = generate_tweets_api(row)
    tweets_list.append(tweets)

    if i % 10 == 0:
        print(f"Processed {i} rows")

df["tweets"] = tweets_list

Processed 0 rows
Processed 10 rows
Processed 20 rows
Processed 30 rows
Processed 40 rows
Processed 50 rows
Processed 60 rows
Processed 70 rows
Processed 80 rows
Processed 90 rows
Processed 100 rows
Processed 110 rows
Processed 120 rows
Processed 130 rows
Processed 140 rows
Processed 150 rows
Processed 160 rows
Processed 170 rows
Processed 180 rows
Processed 190 rows
Processed 200 rows
Processed 210 rows
Processed 220 rows
Processed 230 rows
Processed 240 rows
Processed 250 rows
Processed 260 rows


In [ ]:
df

,Boro,Vol,geometry,congestion,congestion_level,timestamp,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Street Name,tweets
0,Queens,80,POLYGON ((-73.81541389765073 40.74737074495193...,0.035500,0,2024-06-08 22:30:00,New York City Police Department,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,OAK AVENUE,[Just had to navigate around a car taking up t...
1,Bronx,69,POLYGON ((-73.84214963396792 40.85815387505401...,0.122636,0,2025-04-21 11:15:00,New York City Police Department,Blocked Driveway,Partial Access,Street/Sidewalk,EASTCHESTER ROAD,"[My driveway on Eastchester Road is blocked, c..."
2,Manhattan,223,"POLYGON ((-73.9743296272132 40.78160532554132,...",0.294327,0,2025-06-21 12:30:00,New York City Police Department,Illegal Parking,Blocked Bike Lane,Street/Sidewalk,COLUMBUS AVENUE,[I just saw someone blocking the bike lane on ...
3,Manhattan,72,"POLYGON ((-73.9743296272132 40.78160532554132,...",-1.557482,0,2025-06-22 06:30:00,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,COLUMBUS AVENUE,[Just got a parking ticket on Columbus Avenue ...
4,Manhattan,115,POLYGON ((-73.98885407294881 40.77188224822505...,0.523082,0,2024-10-16 15:30:00,Department of Transportation,Street Condition,Pothole,NaN,WEST END AVENUE,"[WEST END AVENUE needs some fixing, a huge pot..."
...,...,...,...,...,...,...,...,...,...,...,...,...
258,Brooklyn,234,POLYGON ((-73.93601997464899 40.72387248144743...,0.721558,0,2024-03-16 13:00:00,Department of Transportation,Traffic Signal Condition,Enclosure Cap,NaN,NaN,[I'm stuck in traffic at the nan street inters...
259,Queens,57,POLYGON ((-73.80240146876282 40.68797242598935...,3.000000,2,2024-01-23 15:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET,[Just saw an abandoned car on Pinegrove St wit...
260,Queens,39,POLYGON ((-73.80240146876282 40.68797242598935...,1.939188,1,2024-01-23 17:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET,[I just saw an abandoned car on Pinegrove Stre...
261,Queens,39,POLYGON ((-73.80240146876282 40.68797242598935...,1.939188,1,2024-01-23 17:30:00,New York City Police Department,Abandoned Vehicle,With License Plate,Street/Sidewalk,PINEGROVE STREET,[I just noticed an abandoned vehicle on Pinegr...


In [ ]:
df.to_csv("tweet2.csv")